**Voice-to-Grasp - additional statistics cells** for `cross_user_predictions.ipynb` (McNemar tests).

# Statistical Analysis Cells — cross_user_predictions.ipynb
Add these cells **after Cell 4 (Summary Table)** in your existing notebook.
They use `all_results` and `summary_df` already computed above.

---
## STAT CELL A · McNemar's Test — Diagonal vs Off-Diagonal Performance
Tests whether the personalised model performs significantly better on its **own user's data**
compared to other users' data (paired binary outcome per prompt).

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar
import itertools

print("=" * 70)
print("McNemar's Test: Model User vs Each Data User")
print("Tests whether classification accuracy differs significantly")
print("between the model's own-user data and other users' data.")
print("=" * 70)

user_ids = list(all_results.keys())

# We need a 'own user' result to compare against each cross-user result.
# MODEL_USER data must be in all_results for this test.
if MODEL_USER not in all_results:
    print(f"NOTE: MODEL_USER '{MODEL_USER}' not in DATA_USERS — McNemar test skipped.")
    print("To enable: add MODEL_USER to DATA_USERS dict in Cell 0 and re-run.")
else:
    own_correct = all_results[MODEL_USER]["correct"].values.astype(int)

    for uid in user_ids:
        if uid == MODEL_USER:
            continue
        other_correct = all_results[uid]["correct"].values.astype(int)
        n = min(len(own_correct), len(other_correct))
        a = own_correct[:n]; b = other_correct[:n]
        # Contingency table: [both correct, own only, other only, neither]
        c00 = int(np.sum((a == 1) & (b == 1)))
        c01 = int(np.sum((a == 1) & (b == 0)))
        c10 = int(np.sum((a == 0) & (b == 1)))
        c11 = int(np.sum((a == 0) & (b == 0)))
        table = [[c00, c01], [c10, c11]]
        result = mcnemar(table, exact=True)
        print(f"\n  {MODEL_USER} (own) vs {uid} (cross):")
        print(f"    Contingency table: {table}")
        print(f"    McNemar statistic = {result.statistic:.4f}")
        print(f"    p-value           = {result.pvalue:.4f}")
        sig = "SIGNIFICANT" if result.pvalue < 0.05 else "not significant"
        print(f"    -> {sig} at alpha = 0.05")

---
## STAT CELL B · Wilcoxon Signed-Rank Test — Diagonal vs Off-Diagonal Accuracy
Formally tests whether the within-user (diagonal) accuracy values in the cross-user matrix
are significantly higher than the off-diagonal (cross-user) values.
This is the key statistical validation of your personalisation claim.

In [ ]:
from scipy.stats import wilcoxon, mannwhitneyu
import warnings

# For this cell you need the full 5x5 cross-user accuracy matrix.
# Replace the values below with those from your Figure 10 matrix.
# Rows = model trained on, Columns = evaluated on
# Format: cross_user_matrix[trained_on][evaluated_on] = accuracy

print("=" * 70)
print("Wilcoxon Signed-Rank Test: Diagonal vs Off-Diagonal Accuracy")
print("=" * 70)

# --- PASTE YOUR CROSS-USER MATRIX VALUES HERE ---
# From Figure 10 in the paper (rows = trained on, cols = evaluated on)
cross_matrix = np.array([
    [0.795, 0.481, 0.667, 0.762, 0.881],   # User_1 model
    [0.421, 0.776, 0.479, 0.667, 0.693],   # User_2 model
    [0.464, 0.464, 0.824, 0.686, 0.836],   # User_3 model
    [0.376, 0.448, 0.481, 0.852, 0.886],   # User_4 model
    [0.462, 0.486, 0.619, 0.776, 0.967],   # User_5 model
])

n = cross_matrix.shape[0]
diagonal     = [cross_matrix[i, i] for i in range(n)]
off_diagonal = [cross_matrix[i, j] for i in range(n) for j in range(n) if i != j]

print(f"\n  Diagonal (within-user) accuracies : {[round(d,3) for d in diagonal]}")
print(f"  Mean diagonal                     : {np.mean(diagonal):.4f}")
print(f"  Off-diagonal accuracies (n={len(off_diagonal)}): {[round(o,3) for o in off_diagonal]}")
print(f"  Mean off-diagonal                 : {np.mean(off_diagonal):.4f}")

# Mann-Whitney U (diagonal vs off-diagonal; unpaired)
stat_mw, p_mw = mannwhitneyu(diagonal, off_diagonal, alternative="greater")
print(f"\n  Mann-Whitney U test (diagonal > off-diagonal):")
print(f"    U = {stat_mw:.4f},  p = {p_mw:.4f}")
print(f"    -> {'SIGNIFICANT' if p_mw < 0.05 else 'not significant'} at alpha = 0.05")

# Effect size: rank-biserial correlation
n1, n2 = len(diagonal), len(off_diagonal)
r_rb = 1 - (2 * stat_mw) / (n1 * n2)
print(f"    Rank-biserial correlation r = {r_rb:.4f} (effect size)")

---
## STAT CELL C · 95% Wilson Confidence Intervals — All Users (Enhanced Summary)
Adds confidence intervals to your existing summary table and prints
a clean, paper-ready results table.

In [ ]:
from statsmodels.stats.proportion import proportion_confint

print("=" * 70)
print("Cross-User Accuracy with 95% Wilson Confidence Intervals")
print("=" * 70)

fmt = "{:<12} {:>10} {:>14} {:>8} {:>8}"
print(fmt.format("Data User", "Accuracy", "95% CI", "Correct", "Total"))
print("-" * 70)

accs = []
for user_id, df in all_results.items():
    n_correct = int(df["correct"].sum())
    n_total   = len(df)
    acc       = n_correct / n_total
    lo, hi    = proportion_confint(n_correct, n_total, method="wilson")
    accs.append(acc)
    print(fmt.format(
        user_id,
        f"{acc:.4f}",
        f"[{lo:.4f}, {hi:.4f}]",
        str(n_correct),
        str(n_total)
    ))

print("-" * 70)
print(fmt.format("Mean", f"{np.mean(accs):.4f}", "", "", ""))
print(fmt.format("SD",   f"{np.std(accs):.4f}",  "", "", ""))
print("=" * 70)

---
## STAT CELL D · Cohen's Kappa — Per-User Classification Agreement
Measures how much better the model performs than chance for each user,
accounting for class imbalance. Values > 0.6 indicate substantial agreement.

In [ ]:
from sklearn.metrics import cohen_kappa_score

print("=" * 70)
print("Cohen's Kappa — Model Agreement vs Ground Truth (Per User)")
print("Interpretation: <0.2 slight | 0.2-0.4 fair | 0.4-0.6 moderate")
print("                0.6-0.8 substantial | >0.8 almost perfect")
print("=" * 70)

fmt = "{:<12} {:>10} {:>12}"
print(fmt.format("Data User", "Kappa", "Interpretation"))
print("-" * 40)

kappas = []
for user_id, df in all_results.items():
    kappa = cohen_kappa_score(df["true_grasp"], df["predicted_grasp"])
    kappas.append(kappa)
    if kappa > 0.8:   interp = "Almost perfect"
    elif kappa > 0.6: interp = "Substantial"
    elif kappa > 0.4: interp = "Moderate"
    elif kappa > 0.2: interp = "Fair"
    else:             interp = "Slight"
    print(fmt.format(user_id, f"{kappa:.4f}", interp))

print("-" * 40)
print(fmt.format("Mean", f"{np.mean(kappas):.4f}", ""))
print(fmt.format("SD",   f"{np.std(kappas):.4f}",  ""))